# Explore LandIQ for any survey year (ZIP → shapefile → tomato codes)

Project survey years: **2018–2024** (see `src/landiq/years/registry.py`). Same logic as [`01_2018_explore_shapefile_and_tomato_codes.ipynb`](../2018/01_2018_explore_shapefile_and_tomato_codes.ipynb), parameterized by **`YEAR`**.

1. Set **`YEAR`** and **`TOMATO_CODES`** (from that year’s DWR / LandIQ legend PDF).
2. Set **`ZIP_FILENAME`** to the exact zip under `data/raw/landiq/<YEAR>/`, or **`None`** to auto-detect (`find_landiq_crop_zip`).
3. Run all cells: extract → load → scan all columns → scan **CROPTYP*** slots → optional PDF.

Hints for zip names and codes: [`configs/landiq/years/`](../../../../configs/landiq/years/) (per-year `*.example.yaml`).

**Filtering:** use [`02_filter_tomato_polygons.ipynb`](../2018/02_filter_tomato_polygons.ipynb) after updating `paths.local.yaml` (set `landiq.year` to match) — tomato rows keep **all** attributes.

## Parameters (edit here)

In [ ]:
from src.landiq.years.registry import SURVEY_YEARS, TOMATO_CODES_DEFAULT, suggested_zip_filename

print("Survey years in this project:", SURVEY_YEARS)

YEAR = 2018  # change to 2019, 2020, …
ZIP_FILENAME = suggested_zip_filename(YEAR)  # None → auto-detect in raw folder
TOMATO_CODES = TOMATO_CODES_DEFAULT  # confirm in that year’s legend PDF
LEGEND_PDF_GLOB = "*Legend*.pdf"

## Imports, paths, extract, load

In [ ]:
from pathlib import Path

import geopandas as gpd
import pandas as pd
from IPython.display import display

from src.landiq.legend_codes import (
    attribute_table_overview,
    croptyp_column_names,
    scan_columns_for_codes,
    scan_croptyp_columns_for_codes,
    summarize_tomato_croptyp_coverage,
    tomato_mask_any_croptyp,
)
from src.landiq.zip_extract import (
    extract_zip,
    find_landiq_crop_zip,
    find_shapefiles,
    pick_main_shapefile,
)
from src.utils.paths import REPO_ROOT

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 120)

YEAR_STR = str(YEAR)
RAW_YEAR = REPO_ROOT / "data" / "raw" / "landiq" / YEAR_STR
EXTRACT_DIR = REPO_ROOT / "data" / "derived" / "landiq_extracted" / YEAR_STR

ZIP_PATH = find_landiq_crop_zip(RAW_YEAR, zip_filename=ZIP_FILENAME)
print("ZIP:", ZIP_PATH)

pdfs = sorted(RAW_YEAR.glob(LEGEND_PDF_GLOB)) if LEGEND_PDF_GLOB else []
LEGEND_PDF = pdfs[0] if pdfs else None
print("Legend PDF:", LEGEND_PDF)

extract_zip(ZIP_PATH, EXTRACT_DIR, clear=True)
print("Shapefiles:", len(find_shapefiles(EXTRACT_DIR)))
shp_main = pick_main_shapefile(EXTRACT_DIR, prefer_name_contains="crop")
print("Using:", shp_main)

gdf = gpd.read_file(shp_main)
print("Rows:", len(gdf), "CRS:", gdf.crs)
gdf.head()

## Attribute overview

In [ ]:
display(attribute_table_overview(gdf))

## Scan all columns for tomato codes

In [ ]:
hits = scan_columns_for_codes(gdf, TOMATO_CODES)
if hits.empty:
    print("No exact matches for", TOMATO_CODES)
else:
    display(hits.sort_values(["column", "code"]))

## All `CROPTYP*` slots (recommended before filtering)

In [ ]:
croptyp_cols = croptyp_column_names(gdf)
print("CROPTYP* columns:", croptyp_cols)
display(summarize_tomato_croptyp_coverage(gdf, TOMATO_CODES, columns=croptyp_cols))
display(scan_croptyp_columns_for_codes(gdf, TOMATO_CODES))
mask_any = tomato_mask_any_croptyp(gdf, TOMATO_CODES, columns=croptyp_cols)
print("Polygons with any tomato code in CROPTYP slots:", int(mask_any.sum()))
for code in TOMATO_CODES:
    m = tomato_mask_any_croptyp(gdf, [code], columns=croptyp_cols)
    print(f"  {code!r} in any slot: {int(m.sum())}")

## Optional: legend PDF text

In [ ]:
try:
    from pypdf import PdfReader
except ImportError:
    print("pip install pypdf")
else:
    if LEGEND_PDF and LEGEND_PDF.is_file():
        reader = PdfReader(str(LEGEND_PDF))
        text = "\n".join(page.extract_text() or "" for page in reader.pages[:8])
        for token in TOMATO_CODES + ("tomato", "Tomato"):
            print(token, "->", text.lower().count(token.lower()), "mentions (first 8 pages)")
        print(text[:2500])
    else:
        print("No PDF matched:", LEGEND_PDF_GLOB, "in", RAW_YEAR)

## Next steps

1. Copy settings into `configs/paths.local.yaml`: `landiq.year`, `landiq.tomato_values` (list), `landiq.crop_columns` (e.g. all `CROPTYP1`–`3`), optional `landiq.zip_filename` / `landiq.output_filename`.
2. Run [`../2018/02_filter_tomato_polygons.ipynb`](../2018/02_filter_tomato_polygons.ipynb) — output is **tomato-only rows** with **full** LandIQ attributes (adjust `landiq.year` in config for other survey years).